# Import Required Libraries

Import the libraries required for loading the trained model, preprocessing artifacts, and generating movie recommendations.

In [50]:
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Layer

In [51]:
class L2Normalization(Layer):
    """
    Normalizes each embedding vector to have unit L2 norm.
    """

    def call(self, inputs):
        return tf.linalg.l2_normalize(inputs, axis=1)

# Load Trained Model

Load the saved Two-Tower recommendation model.

In [52]:
model = tf.keras.models.load_model(
    "../artifacts/two_tower_model.keras",
    custom_objects={
        "L2Normalization": L2Normalization
    }
)

print("Model loaded successfully.")

Model loaded successfully.


# Load Mapping Dictionaries

Load the user and movie index mappings created during training.

In [53]:
with open("../artifacts/user_to_index.pkl", "rb") as f:
    user_to_index = pickle.load(f)

with open("../artifacts/movie_to_index.pkl", "rb") as f:
    movie_to_index = pickle.load(f)

with open("../artifacts/index_to_movie.pkl", "rb") as f:
    index_to_movie = pickle.load(f)

print("Mappings loaded successfully.")

Mappings loaded successfully.


# Load Movie Metadata

Load the movie dataset to map movie IDs to movie titles.

In [54]:
movies = pd.read_csv("../data/movies.csv")

movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


# Verify Loaded Model

Display the architecture of the loaded recommendation model.

In [55]:
model.summary()

Model: "TwoTowerRecommender"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ user_input          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ movie_input         │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_embedding      │ (None, 1, 32)     │    193,280 │ user_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ movie_embedding     │ (None, 1, 32)     │    118,592 │ movie_input[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_flatten        │ (None, 32)        │          0 │ user_embedding[0… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ movie_flatten       │ (None, 32)        │          0 │ movie_embedding[… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ UserTower           │ (None, 32)        │     45,472 │ user_flatten[0][… │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ MovieTower          │ (None, 32)        │     45,472 │ movie_flatten[0]… │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_normalization  │ (None, 32)        │          0 │ UserTower[0][0]   │
│ (L2Normalization)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ movie_normalization │ (None, 32)        │          0 │ MovieTower[0][0]  │
│ (L2Normalization)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ similarity_score    │ (None, 1)         │          0 │ user_normalizati… │
│ (Dot)               │                   │            │ movie_normalizat… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,208,450 (4.61 MB)

 Trainable params: 402,816 (1.54 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 805,634 (3.07 MB)

# Select a User

Choose a user for whom movie recommendations will be generated.

In [56]:
user_id = 1

# Encode User ID

Convert the original user ID into the encoded user index used by the model.

In [57]:
user_index = user_to_index[user_id]

print(user_index)

0


# Generate Recommendation Scores

Predict the similarity score between the selected user and every movie in the dataset.

In [58]:
# Total number of movies
num_movies = len(movie_to_index)

# Same user repeated for every movie
user_input = np.full(
    shape = num_movies,
    fill_value = user_index
)

# Movie indices
movie_input = np.arange(num_movies)

# Predict similarity scores
scores = model.predict(
    [user_input, movie_input],
    verbose = 0
).flatten()

print(scores[:10])

[0.8924961  0.7083103  0.77731156 0.65013736 0.7760898  0.75959724
 0.7414065  0.75815743 0.63890964 0.8009568 ]


# Retrieve Top Movies

Sort the predicted similarity scores and retrieve the highest-ranked movies.

In [59]:
top_indices = np.argsort(scores)[::-1][:10]

top_indices

array([ 390, 1025,  871, 3669,  466,  583, 2213,  573,  443, 1253])

# Decode Movie Indices

Convert encoded movie indices back to their original MovieLens movie IDs.

In [60]:
recommended_movie_ids = [
    index_to_movie[idx]
    for idx in top_indices
]

recommended_movie_ids

[np.int64(404),
 np.int64(1097),
 np.int64(932),
 np.int64(3916),
 np.int64(480),
 np.int64(597),
 np.int64(2406),
 np.int64(587),
 np.int64(457),
 np.int64(1351)]

# Display Recommended Movies

Display the titles of the recommended movies ranked by predicted similarity score.

In [61]:
recommendations = pd.DataFrame({
    "movieId": recommended_movie_ids,
    "score": scores[top_indices]
})

recommendations = (
    recommendations
    .merge(
        movies,
        on="movieId"
    )
    .sort_values(
        by="score",
        ascending=False
    )
    .reset_index(drop=True)
)

recommendations

,movieId,score,title,genres
0,404,0.940557,Brother Minister: The Assassination of Malcolm...,Documentary
1,1097,0.929861,E.T. the Extra-Terrestrial (1982),Children's|Drama|Fantasy|Sci-Fi
2,932,0.921956,"Affair to Remember, An (1957)",Romance
3,3916,0.914108,Remember the Titans (2000),Drama
4,480,0.911917,Jurassic Park (1993),Action|Adventure|Sci-Fi
5,597,0.911653,Pretty Woman (1990),Comedy|Romance
6,2406,0.910818,Romancing the Stone (1984),Action|Adventure|Comedy|Romance
7,587,0.909437,Ghost (1990),Comedy|Romance|Thriller
8,457,0.909391,"Fugitive, The (1993)",Action|Thriller
9,1351,0.909330,Blood & Wine (1997),Drama


# Load User Ratings

Load the ratings dataset to identify movies that the selected user has already watched.

In [62]:
ratings = pd.read_csv("../data/ratings.csv")

ratings.head()

,userId,movieId,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


# Identify Watched Movies

Retrieve all movies that have already been rated by the selected user.

In [63]:
watched_movie_ids = ratings.loc[
    ratings["userId"] == user_id,
    "movieId"
].unique()

print(f"Movies already watched: {len(watched_movie_ids)}")

Movies already watched: 53


# Filter Candidate Movies

Remove movies that the user has already watched so that only unseen movies are considered for recommendation.

In [64]:
candidate_movies = movies[
    ~movies["movieId"].isin(watched_movie_ids)
].copy()

print(f"Candidate movies: {len(candidate_movies)}")

Candidate movies: 3830


# Encode Candidate Movies

Convert candidate movie IDs into encoded movie indices used by the recommendation model.

In [65]:
candidate_movies["movie_index"] = candidate_movies["movieId"].map(
    movie_to_index
)

candidate_movies.head()

,movieId,title,genres,movie_index
1,2,Jumanji (1995),Adventure|Children's|Fantasy,1.0
2,3,Grumpier Old Men (1995),Comedy|Romance,2.0
3,4,Waiting to Exhale (1995),Comedy|Drama,3.0
4,5,Father of the Bride Part II (1995),Comedy,4.0
5,6,Heat (1995),Action|Crime|Thriller,5.0


# Remove Invalid Movies

Drop movies that do not have a valid encoded index.

In [68]:
candidate_movies = candidate_movies[
    candidate_movies["movie_index"].notna()
].copy()

candidate_movies["movie_index"] = (
    candidate_movies["movie_index"]
    .astype(int)
)

# Predict Recommendation Scores

Predict similarity scores for all unseen candidate movies.

In [69]:
movie_indices = candidate_movies["movie_index"].astype(int).values

user_indices = np.full(
    len(movie_indices),
    user_index
)

scores = model.predict(
    [user_indices, movie_indices],
    verbose = 0
).flatten()

candidate_movies["score"] = scores

# Top Movie Recommendations

Rank candidate movies by predicted similarity score and retrieve the top recommendations.

In [70]:
top_recommendations = (
    candidate_movies
    .sort_values(
        by="score",
        ascending=False
    )
    .head(10)
    .reset_index(drop=True)
)

top_recommendations[
    [
        "movieId",
        "title",
        "genres",
        "score"
    ]
]

,movieId,title,genres,score
0,404,Brother Minister: The Assassination of Malcolm...,Documentary,0.940557
1,932,"Affair to Remember, An (1957)",Romance,0.921956
2,3916,Remember the Titans (2000),Drama,0.914108
3,480,Jurassic Park (1993),Action|Adventure|Sci-Fi,0.911917
4,597,Pretty Woman (1990),Comedy|Romance,0.911653
5,2406,Romancing the Stone (1984),Action|Adventure|Comedy|Romance,0.910818
6,587,Ghost (1990),Comedy|Romance|Thriller,0.909437
7,457,"Fugitive, The (1993)",Action|Thriller,0.909391
8,1351,Blood & Wine (1997),Drama,0.909330
9,1380,Grease (1978),Comedy|Musical|Romance,0.909121
